# Model Training Test Notebook

This notebook tests model training end-to-end for resume classification.

In [8]:
from pathlib import Path
import sys
import importlib
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'tests' else Path.cwd()
src_path = project_root / 'src'
dataset_dir = project_root / 'Dataset'

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from data_loader import load_train_test_data
from preprocessing import preprocess_dataframe
from feature_extraction import extract_train_test_features

import model_training as model_training_module
importlib.reload(model_training_module)

get_model_candidates = model_training_module.get_model_candidates
compare_models = model_training_module.compare_models
get_best_model = model_training_module.get_best_model
compare_model_feature_combinations = model_training_module.compare_model_feature_combinations
get_best_model_feature_combination = model_training_module.get_best_model_feature_combination

print('Project root :', project_root)
print('Dataset path :', dataset_dir)
print('Train exists :', (dataset_dir / 'train.csv').exists())
print('Test exists  :', (dataset_dir / 'test.csv').exists())

Project root : c:\Users\POOJA\OneDrive\Desktop\project
Dataset path : c:\Users\POOJA\OneDrive\Desktop\project\Dataset
Train exists : True
Test exists  : True


## 1) Load and preprocess data

In [9]:
train_df, test_df = load_train_test_data(dataset_dir)
train_processed = preprocess_dataframe(train_df, text_column='text')
test_processed = preprocess_dataframe(test_df, text_column='text')

print('Train shape:', train_processed.shape)
print('Test shape :', test_processed.shape)
display(train_processed.head(2))

Train shape: (1986, 2)
Test shape : (497, 2)


,text,label
0,engin consult profession summari deliv valu pr...,CONSULTANT
1,carpent summari carpent foreman posit effect u...,CONSTRUCTION


## 2) Build TF-IDF features

In [10]:
vectorizer, x_train, x_test = extract_train_test_features(
    train_processed,
    test_processed,
    text_column='text',
    max_features=10000,
    ngram_range=(1, 3),
    min_df=1,
    max_df=0.9,
    analyzer='word',
    stop_words='english',
)

y_train = train_processed['label']
y_test = test_processed['label']

print('x_train shape:', x_train.shape)
print('x_test shape :', x_test.shape)
print('Vocabulary size used:', len(vectorizer.vocabulary_))

C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


x_train shape: (1986, 10000)
x_test shape : (497, 10000)
Vocabulary size used: 10000


## 3) Compare models

In [11]:
models = get_model_candidates()
results_df, trained_models = compare_models(x_train, y_train, x_test, y_test, models=models)

display_columns = [
    'model',
    'accuracy',
    'balanced_accuracy',
    'precision',
    'recall',
    'f1_score',
]
results_df[display_columns]

C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


,model,accuracy,balanced_accuracy,precision,recall,f1_score
0,RandomForest,0.734406,0.688004,0.696501,0.688004,0.662208
1,LinearSVM,0.726358,0.678750,0.697596,0.678750,0.665210
2,RidgeClassifier,0.720322,0.676215,0.698702,0.676215,0.661881
3,PassiveAggressive,0.714286,0.670060,0.694456,0.670060,0.657231
4,LogisticRegression,0.698189,0.655772,0.671714,0.655772,0.640477
5,ComplementNB,0.589537,0.560926,0.599729,0.560926,0.518334
6,MultinomialNB,0.563380,0.524907,0.518324,0.524907,0.499527
7,KNN,0.531187,0.506188,0.535803,0.506188,0.491299


## 4) Best model

In [12]:
best_name, best_model, best_results_df = get_best_model(x_train, y_train, x_test, y_test)
print('Best model:', best_name)
display(best_results_df.head(3))

C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


Best model: RandomForest


,model,accuracy,balanced_accuracy,precision,recall,f1_score,precision_weighted,recall_weighted,f1_weighted
0,RandomForest,0.734406,0.688004,0.696501,0.688004,0.662208,0.749559,0.734406,0.709816
1,LinearSVM,0.726358,0.678750,0.697596,0.678750,0.665210,0.725119,0.726358,0.711541
2,RidgeClassifier,0.720322,0.676215,0.698702,0.676215,0.661881,0.722813,0.720322,0.704631


## 5) Compare model + feature combinations

In [13]:
feature_settings = [
    {'max_features': 10000, 'ngram_range': (1, 2), 'min_df': 1, 'max_df': 0.95, 'analyzer': 'word', 'stop_words': 'english'},
    {'max_features': 10000, 'ngram_range': (1, 3), 'min_df': 2, 'max_df': 0.9, 'analyzer': 'word', 'stop_words': 'english'},
    {'max_features': 10000, 'ngram_range': (3, 5), 'min_df': 2, 'max_df': 1.0, 'analyzer': 'char_wb', 'stop_words': None},
]

combo_df = compare_model_feature_combinations(
    train_processed,
    test_processed,
    text_column='text',
    label_column='label',
    feature_settings=feature_settings,
)

combo_columns = [
    'model',
    'accuracy',
    'balanced_accuracy',
    'precision',
    'recall',
    'f1_score',
    'max_features',
    'ngram_range',
    'min_df',
    'max_df',
    'analyzer',
    'stop_words',
]
combo_df[combo_columns].head(10)

C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(
C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(
C:\Users\POOJA\AppData\Local\Python\py

,model,accuracy,balanced_accuracy,precision,recall,f1_score,max_features,ngram_range,min_df,max_df,analyzer,stop_words
0,RandomForest,0.738431,0.694935,0.725747,0.694935,0.674546,10000,"(1, 3)",2,0.90,word,english
1,RidgeClassifier,0.732394,0.687546,0.709840,0.687546,0.673964,10000,"(1, 3)",2,0.90,word,english
2,RidgeClassifier,0.730382,0.692485,0.726897,0.692485,0.682005,10000,"(1, 2)",1,0.95,word,english
3,RandomForest,0.724346,0.684384,0.720765,0.684384,0.665877,10000,"(1, 2)",1,0.95,word,english
4,LinearSVM,0.724346,0.676666,0.695700,0.676666,0.663474,10000,"(1, 3)",2,0.90,word,english
5,PassiveAggressive,0.716298,0.670265,0.697119,0.670265,0.658206,10000,"(1, 2)",1,0.95,word,english
6,PassiveAggressive,0.714286,0.670060,0.694344,0.670060,0.657171,10000,"(1, 3)",2,0.90,word,english
7,LinearSVM,0.712274,0.667458,0.686921,0.667458,0.653474,10000,"(1, 2)",1,0.95,word,english
8,LogisticRegression,0.698189,0.655574,0.672855,0.655574,0.641424,10000,"(1, 3)",2,0.90,word,english
9,LogisticRegression,0.686117,0.647961,0.666053,0.647961,0.632664,10000,"(1, 2)",1,0.95,word,english


## 5.1) Compare Model + TF-IDF (best setting per model)

In [14]:
best_per_model_df = (
    combo_df.sort_values(['model', 'accuracy'], ascending=[True, False])
    .drop_duplicates(subset=['model'])
    .sort_values('accuracy', ascending=False)
    .reset_index(drop=True)
)

best_per_model_columns = [
    'model',
    'accuracy',
    'balanced_accuracy',
    'precision',
    'recall',
    'f1_score',
    'max_features',
    'ngram_range',
    'min_df',
    'max_df',
    'analyzer',
    'stop_words',
]

best_per_model_df[best_per_model_columns]

,model,accuracy,balanced_accuracy,precision,recall,f1_score,max_features,ngram_range,min_df,max_df,analyzer,stop_words
0,RandomForest,0.738431,0.694935,0.725747,0.694935,0.674546,10000,"(1, 3)",2,0.90,word,english
1,RidgeClassifier,0.732394,0.687546,0.709840,0.687546,0.673964,10000,"(1, 3)",2,0.90,word,english
2,LinearSVM,0.724346,0.676666,0.695700,0.676666,0.663474,10000,"(1, 3)",2,0.90,word,english
3,PassiveAggressive,0.716298,0.670265,0.697119,0.670265,0.658206,10000,"(1, 2)",1,0.95,word,english
4,LogisticRegression,0.698189,0.655574,0.672855,0.655574,0.641424,10000,"(1, 3)",2,0.90,word,english
5,ComplementNB,0.585513,0.557240,0.598002,0.557240,0.514968,10000,"(1, 3)",2,0.90,word,english
6,MultinomialNB,0.565392,0.528841,0.527037,0.528841,0.504089,10000,"(1, 2)",1,0.95,word,english
7,KNN,0.531187,0.505876,0.534999,0.505876,0.490551,10000,"(1, 3)",2,0.90,word,english


## 6) Best combination summary

In [15]:
best_combo = get_best_model_feature_combination(
    train_processed,
    test_processed,
    text_column='text',
    label_column='label',
    feature_settings=feature_settings,
)
pd.DataFrame([best_combo])

C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(
C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)
C:\Users\POOJA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:2043: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(
C:\Users\POOJA\AppData\Local\Python\py

,model,accuracy,balanced_accuracy,precision,recall,f1_score,precision_weighted,recall_weighted,f1_weighted,max_features,ngram_range,min_df,max_df,analyzer,stop_words
0,RandomForest,0.738431,0.694935,0.725747,0.694935,0.674546,0.750316,0.738431,0.716468,10000,"(1, 3)",2,0.9,word,english
